### 2d
- id lang + translate > fit into pipe
- test sentiment
- test commitment
- test many reports (1 click pipeline!)
- scores 
    - specificity: pdf-wide or add highest
    - Validate scoring by inspecting chunks
    - calc talk-score

In [26]:
"""
Optimized pipeline for large-scale climate report analysis
Features:
- Pre-filtering with climate detector
- Batch processing
- Caching preprocessed data
- Multi-stage analysis
"""

import os
import json
import pickle
from pathlib import Path
from typing import List, Dict
import pdfplumber
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from tqdm.auto import tqdm

class ClimateReportAnalyzer:
    # Configuration constants
    MIN_CHUNK_LENGTH = 200  # Minimum characters for a chunk to be analyzed
    BATCH_SIZE = 48         # Number of chunks to process in parallel

    def __init__(self, cache_dir="cache", use_cache=True, run_climate_filter=True,
                 run_specificity=True, run_sentiment=True, run_commitment=True):
        """
        Initialize analyzer with configuration

        Args:
            cache_dir: Directory for caching results
            use_cache: Use cached text extractions
            run_climate_filter: Pre-filter with climate detector
            run_specificity: Run specificity analysis
            run_sentiment: Run sentiment analysis
            run_commitment: Run commitment detection
        """
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        # Store config
        self.use_cache = use_cache
        self.run_climate_filter = run_climate_filter
        self.run_specificity = run_specificity
        self.run_sentiment = run_sentiment
        self.run_commitment = run_commitment

        self.device = 0 if torch.cuda.is_available() else -1
        print(f"Using device: {'GPU' if self.device == 0 else 'CPU'}")
        print(f"Config: cache={use_cache}, filter={run_climate_filter}, "
              f"spec={run_specificity}, sent={run_sentiment}, commit={run_commitment}")

        self.models = {}

    def load_model(self, model_name: str, task_name: str):
        """Load and cache a model"""
        if task_name not in self.models:
            print(f"Loading {task_name} model...")
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.models[task_name] = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                device=self.device
            )
        return self.models[task_name]

    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract text from PDF with caching"""
        cache_file = self.cache_dir / f"{Path(pdf_path).stem}_text.txt"

        if self.use_cache and cache_file.exists():
            print(f"Loading cached text from {cache_file}")
            return cache_file.read_text(encoding='utf-8')

        print(f"Extracting text from {pdf_path}...")
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in tqdm(pdf.pages, desc="Reading pages"):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        if self.use_cache:
            cache_file.write_text(text, encoding='utf-8')
            print(f"Cached text to {cache_file}")

        return text

    def chunk_text_by_paragraphs(self, text: str, max_tokens=400) -> List[str]:
        """
        Split by paragraphs, merge small ones, split large ones

        Args:
            max_tokens: Maximum tokens per chunk (default 400, model max is 512)

        Returns:
            List of text chunks, each roughly paragraph-sized
        """
        # Rough estimate: 1 token ≈ 4 characters
        MAX_CHARS = max_tokens * 4  # 400 tokens ≈ 1600 chars

        # Step 1: Split by double newline (paragraphs)
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para_size = len(para)
            current_size = len(current_chunk)

            # Case A: Paragraph is way too big → split it by sentences
            if para_size > MAX_CHARS:
                # Save current chunk first
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = ""

                # Split large paragraph into sentences
                sentences = para.split('. ')
                for sent in sentences:
                    if current_size + len(sent) < MAX_CHARS:
                        current_chunk += sent + ". "
                        current_size = len(current_chunk)
                    else:
                        if current_chunk:
                            chunks.append(current_chunk.strip())
                        current_chunk = sent + ". "
                        current_size = len(current_chunk)

            # Case B: Adding this paragraph would exceed limit → save & start new
            elif current_size + para_size > MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = para

            # Case C: Small paragraph → accumulate
            else:
                current_chunk += ("\n\n" if current_chunk else "") + para

        # Don't forget last chunk
        if current_chunk:
            chunks.append(current_chunk.strip())

        # Filter very short chunks
        return [c for c in chunks if len(c) > self.MIN_CHUNK_LENGTH]

    def filter_climate_chunks(self, chunks: List[str], batch_size=None) -> List[Dict]:
        """Filter chunks to keep only climate-related ones"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector",
            "detector"
        )

        print(f"\nFiltering {len(chunks)} chunks for climate content...")
        climate_chunks = []

        # Process in batches for MUCH better performance (10-20x speedup)
        for i in tqdm(range(0, len(chunks), batch_size), desc="Detecting climate content"):
            batch = chunks[i:i+batch_size]
            try:
                # KEY: Pass batch_size to enable batching in the pipeline
                results = detector(batch, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch, results):
                    # Keep if labeled as "climate" or similar
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'text': chunk,
                            'detector_score': result['score']
                        })
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        print(f"Kept {len(climate_chunks)} climate-related chunks ({len(climate_chunks)/len(chunks)*100:.1f}%)")
        return climate_chunks

    def analyze_specificity(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Analyze climate specificity with batching for performance"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        specificity = self.load_model(
            "climatebert/distilroberta-base-climate-specificity",
            "specificity"
        )

        print(f"\nAnalyzing specificity for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing specificity"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                # BATCHED: 10-20x faster than processing one-by-one
                results = specificity(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['specificity_label'] = result['label']
                    chunk['specificity_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def analyze_sentiment(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Analyze climate sentiment (opportunity/neutral/risk)"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        sentiment = self.load_model(
            "climatebert/distilroberta-base-climate-sentiment",
            "sentiment"
        )

        print(f"\nAnalyzing sentiment for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing sentiment"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                results = sentiment(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['sentiment_label'] = result['label']
                    chunk['sentiment_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def analyze_commitments(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Detect climate commitments and actions"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        commitment = self.load_model(
            "climatebert/distilroberta-base-climate-commitment",
            "commitment"
        )

        print(f"\nAnalyzing commitments for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing commitments"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                results = commitment(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['commitment_label'] = result['label']
                    chunk['commitment_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def calculate_report_score(self, analyzed_chunks: List[Dict]) -> Dict:
        """Calculate overall scores for the report"""
        # Correct labels for climate-specificity model
        score_map = {
            "specific": 1.0,
            "non-specific": 0.0,
        }

        specificity_scores = []
        for chunk in analyzed_chunks:
            label = chunk.get('specificity_label', '').lower()
            confidence = chunk.get('specificity_score', 0.5)

            # Map label to score
            base_score = score_map.get(label, 0.5)  # Default 0.5 if unknown
            weighted_score = base_score * confidence
            specificity_scores.append(weighted_score)

        overall_score = sum(specificity_scores) / len(specificity_scores) if specificity_scores else 0

        # Count labels
        label_dist = {}
        for chunk in analyzed_chunks:
            label = chunk.get('specificity_label', 'unknown')
            label_dist[label] = label_dist.get(label, 0) + 1

        return {
            'overall_specificity': overall_score,
            'num_chunks_analyzed': len(analyzed_chunks),
            'num_climate_chunks': len([c for c in analyzed_chunks if c.get('specificity_label') == 'specific']),
            'label_distribution': label_dist,
            'specificity_scores': specificity_scores
        }

    def process_report(self, pdf_path: str):
        """
        Full pipeline for processing a report
        Uses config flags set in __init__ to determine which analyses to run
        """
        print(f"\n{'='*80}")
        print(f"Processing: {pdf_path}")
        print(f"{'='*80}")

        # Step 1: Extract text
        text = self.extract_text_from_pdf(pdf_path)
        print(f"Total text length: {len(text)} characters")

        # Step 2: Chunk text (by paragraphs)
        chunks = self.chunk_text_by_paragraphs(text)
        print(f"Created {len(chunks)} chunks")

        # Step 3: Filter for climate content (optional)
        if self.run_climate_filter:
            climate_chunks = self.filter_climate_chunks(chunks)
        else:
            climate_chunks = [{'text': c} for c in chunks]

        # Step 4-6: Run enabled analyses
        analyzed_chunks = climate_chunks

        if self.run_specificity:
            analyzed_chunks = self.analyze_specificity(analyzed_chunks)

        if self.run_sentiment:
            analyzed_chunks = self.analyze_sentiment(analyzed_chunks)

        if self.run_commitment:
            analyzed_chunks = self.analyze_commitments(analyzed_chunks)

        # Step 7: Calculate scores
        scores = self.calculate_report_score(analyzed_chunks)

        # Step 8: Save results
        results = {
            'pdf_path': str(pdf_path),
            'total_chunks': len(chunks),
            'climate_chunks': len(climate_chunks),
            'scores': scores,
            'sample_chunks': analyzed_chunks[:10]  # Save first 10 for inspection
        }

        results_file = self.cache_dir / f"{Path(pdf_path).stem}_results.json"
        with open(results_file, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\nSaved results to {results_file}")

        return scores

    def process_all_reports(self, reports_dir: str, pattern="**/*.pdf"):
        """Process all PDFs in directory tree using config from __init__"""
        pdf_files = list(Path(reports_dir).glob(pattern))
        print(f"\nFound {len(pdf_files)} PDF files in {reports_dir}")

        results = []
        failed = []

        for pdf_path in tqdm(pdf_files, desc="Processing reports"):
            try:
                scores = self.process_report(str(pdf_path))
                results.append({
                    'file': str(pdf_path),
                    'company': pdf_path.parent.name,
                    'year': self._extract_year(pdf_path.name),
                    'scores': scores
                })
            except Exception as e:
                print(f"\n❌ Failed: {pdf_path}")
                print(f"   Error: {e}")
                failed.append(str(pdf_path))
                continue

        # Print summary
        self._print_batch_summary(results, failed, len(pdf_files))

        # Save aggregated results
        output_file = self.cache_dir / 'all_results.json'
        with open(output_file, 'w') as f:
            json.dump({
                'results': results,
                'failed': failed,
                'total_processed': len(results),
                'total_found': len(pdf_files)
            }, f, indent=2)
        print(f"Results saved to: {output_file}")

        return results

    def process(self, path: str):
        """
        Smart processor: automatically detects if path is a file or directory
        Uses config from __init__ for all processing options
        """
        path_obj = Path(path)

        if not path_obj.exists():
            raise FileNotFoundError(f"Path not found: {path}")

        # Case 1: Single PDF file
        if path_obj.is_file() and path_obj.suffix.lower() == '.pdf':
            print(f"\n{'='*80}")
            print(f"PROCESSING SINGLE PDF")
            print(f"{'='*80}")

            scores = self.process_report(str(path_obj))
            self._print_single_summary(path_obj, scores)
            return scores

        # Case 2: Directory
        elif path_obj.is_dir():
            nested_pdfs = list(path_obj.glob("**/*.pdf"))

            if not nested_pdfs:
                print(f"❌ No PDF files found in {path}")
                return []

            print(f"\n{'='*80}")
            print(f"PROCESSING DIRECTORY: {path}")
            print(f"{'='*80}")
            print(f"Found {len(nested_pdfs)} PDF files")

            results = self.process_all_reports(str(path_obj), pattern="**/*.pdf")
            return results

        else:
            raise ValueError(f"Path must be a PDF file or directory: {path}")

    def _extract_year(self, filename: str) -> str:
        """Extract year from filename (e.g., Report_2020.pdf -> 2020)"""
        import re
        match = re.search(r'20\d{2}', filename)
        return match.group(0) if match else 'unknown'

    def _print_single_summary(self, pdf_path: Path, scores: Dict):
        """Pretty print results for single PDF"""
        print("\n" + "="*80)
        print("FINAL RESULTS")
        print("="*80)
        print(f"File: {pdf_path.name}")
        print(f"Company: {pdf_path.parent.name}")

        if 'talk_score' in scores:
            print(f"\n🎯 TALK SCORE: {scores['talk_score']:.3f}")
            print(f"  → {self._interpret_talk_score(scores['talk_score'])}")

        print(f"\n📊 Component Scores:")
        if 'overall_specificity' in scores:
            print(f"  Specificity: {scores['overall_specificity']:.3f} - {self._interpret_score(scores['overall_specificity'])}")

        if 'overall_commitment' in scores:
            print(f"  Commitment:  {scores['overall_commitment']:.3f}")

        if 'overall_sentiment' in scores:
            print(f"  Sentiment:   {scores['overall_sentiment']:.3f}")

        print(f"\nChunks analyzed: {scores['num_chunks_analyzed']}")

        # Show distributions
        if 'specificity_distribution' in scores:
            print(f"\nSpecificity Distribution:")
            for label, count in sorted(scores['specificity_distribution'].items()):
                pct = count / scores['num_chunks_analyzed'] * 100
                bar = '█' * int(pct / 2)
                print(f"  {label:15s}: {count:3d} ({pct:5.1f}%) {bar}")

        if 'sentiment_distribution' in scores:
            print(f"\nSentiment Distribution:")
            for label, count in sorted(scores['sentiment_distribution'].items()):
                pct = count / scores['num_chunks_analyzed'] * 100
                bar = '█' * int(pct / 2)
                print(f"  {label:15s}: {count:3d} ({pct:5.1f}%) {bar}")

    def _interpret_talk_score(self, score: float) -> str:
        """Interpret combined talk score"""
        if score >= 0.7:
            return "Strong climate communication (concrete & committed)"
        elif score >= 0.5:
            return "Moderate climate communication (some specifics & commitments)"
        elif score >= 0.3:
            return "Weak climate communication (vague & lacks commitments)"
        else:
            return "Very weak climate communication (lacks substance)"

    def _print_batch_summary(self, results: List[Dict], failed: List[str], total: int):
        """Pretty print results for batch processing"""
        print(f"\n{'='*80}")
        print(f"BATCH PROCESSING SUMMARY")
        print(f"{'='*80}")
        print(f"✅ Successfully processed: {len(results)}/{total}")

        if failed:
            print(f"❌ Failed: {len(failed)}")
            for f in failed[:5]:  # Show first 5
                print(f"   - {Path(f).name}")
            if len(failed) > 5:
                print(f"   ... and {len(failed)-5} more")

        if results:
            # Calculate aggregate statistics
            avg_score = sum(r['scores']['overall_specificity'] for r in results) / len(results)
            print(f"\nAggregate Statistics:")
            print(f"  Average Specificity Score: {avg_score:.3f}")
            print(f"  → {self._interpret_score(avg_score)}")

            # Top 5 and bottom 5
            sorted_results = sorted(results, key=lambda x: x['scores']['overall_specificity'], reverse=True)

            print(f"\n📊 Top 5 Most Specific Reports:")
            for i, r in enumerate(sorted_results[:5], 1):
                score = r['scores']['overall_specificity']
                print(f"  {i}. {r['company']:20s} ({r.get('year', '?')}): {score:.3f}")

            print(f"\n📊 Top 5 Least Specific Reports:")
            for i, r in enumerate(sorted_results[-5:], 1):
                score = r['scores']['overall_specificity']
                print(f"  {i}. {r['company']:20s} ({r.get('year', '?')}): {score:.3f}")

    def _interpret_score(self, score: float) -> str:
        """Interpret specificity score"""
        if score >= 0.7:
            return "Highly specific (concrete targets & actions)"
        elif score >= 0.5:
            return "Moderately specific (mix of concrete & vague)"
        elif score >= 0.3:
            return "Low specificity (mostly vague statements)"
        else:
            return "Very low specificity (lacks concrete information)"

    def inspect_chunks_interactive(self, pdf_path: str, mode="random", n=3,
                                  min_confidence=0.0, max_confidence=1.0,
                                  label_filter=None):
        """
        Advanced chunk inspection for manual evaluation

        Args:
            pdf_path: Path to PDF
            mode: "random", "high_conf", "low_conf", "high_score", "low_score"
            n: Number of chunks to show
            min_confidence: Minimum confidence threshold
            max_confidence: Maximum confidence threshold
            label_filter: Filter by label (e.g., "specific", "opportunity")

        Examples:
            # Random sampling for general inspection
            analyzer.inspect_chunks_interactive(pdf, mode="random", n=5)

            # High confidence specific chunks (easy to verify)
            analyzer.inspect_chunks_interactive(pdf, mode="high_conf",
                                               label_filter="specific", n=3)

            # Low confidence chunks (where model is uncertain)
            analyzer.inspect_chunks_interactive(pdf, mode="low_conf", n=5)

            # Chunks scored as high specificity
            analyzer.inspect_chunks_interactive(pdf, mode="high_score", n=3)
        """
        import random

        # Load results
        results_file = self.cache_dir / f"{Path(pdf_path).stem}_results.json"
        if not results_file.exists():
            print(f"No results found. Run process() first.")
            return

        with open(results_file) as f:
            data = json.load(f)

        chunks = data.get('sample_chunks', [])
        if not chunks:
            print("No chunks available. Re-run analysis.")
            return

        # Filter by label if specified
        if label_filter:
            chunks = [c for c in chunks if label_filter.lower() in
                     str(c.get('specificity_label', '')).lower() or
                     label_filter.lower() in str(c.get('sentiment_label', '')).lower()]

        # Filter by confidence range
        chunks = [c for c in chunks if min_confidence <= c.get('specificity_score', 0.5) <= max_confidence]

        if not chunks:
            print(f"No chunks matching criteria (label={label_filter}, conf={min_confidence}-{max_confidence})")
            return

        # Select chunks based on mode
        if mode == "random":
            selected = random.sample(chunks, min(n, len(chunks)))
        elif mode == "high_conf":
            sorted_chunks = sorted(chunks, key=lambda x: x.get('specificity_score', 0), reverse=True)
            selected = sorted_chunks[:n]
        elif mode == "low_conf":
            sorted_chunks = sorted(chunks, key=lambda x: x.get('specificity_score', 1))
            selected = sorted_chunks[:n]
        elif mode == "high_score":
            # High score = specific label with high confidence
            sorted_chunks = sorted(chunks,
                                 key=lambda x: 1.0 if x.get('specificity_label','').lower() in ['spec','specific']
                                 else 0.0 * x.get('specificity_score', 0),
                                 reverse=True)
            selected = sorted_chunks[:n]
        elif mode == "low_score":
            sorted_chunks = sorted(chunks,
                                 key=lambda x: 0.0 if x.get('specificity_label','').lower() in ['non','non-specific']
                                 else 1.0 * x.get('specificity_score', 0))
            selected = sorted_chunks[:n]
        else:
            selected = chunks[:n]

        # Display
        print(f"\n{'='*80}")
        print(f"CHUNK INSPECTION - Mode: {mode}")
        print(f"PDF: {Path(pdf_path).name}")
        print(f"Showing {len(selected)} of {len(chunks)} chunks")
        print(f"{'='*80}\n")

        for i, chunk in enumerate(selected, 1):
            print(f"{'─'*80}")
            print(f"Chunk #{i}")
            print(f"{'─'*80}")

            # Show all scores
            spec_label = chunk.get('specificity_label', 'N/A')
            spec_conf = chunk.get('specificity_score', 0)
            print(f"Specificity: {spec_label} (confidence: {spec_conf:.3f})")

            if 'sentiment_label' in chunk:
                sent_label = chunk.get('sentiment_label')
                sent_conf = chunk.get('sentiment_score', 0)
                print(f"Sentiment:   {sent_label} (confidence: {sent_conf:.3f})")

            if 'commitment_label' in chunk:
                commit_label = chunk.get('commitment_label')
                commit_conf = chunk.get('commitment_score', 0)
                print(f"Commitment:  {commit_label} (confidence: {commit_conf:.3f})")

            if 'detector_score' in chunk:
                det_conf = chunk.get('detector_score')
                print(f"Climate relevance: {det_conf:.3f}")

            text = chunk.get('text', '')
            print(f"\nText ({len(text)} chars):")
            print(f"{text[:500]}{'...' if len(text) > 500 else ''}\n")

    def show_chunk_structure(self, pdf_path: str, stage="all"):
        """
        Show chunk structure at different pipeline stages

        Args:
            pdf_path: Path to PDF
            stage: "raw", "filtered", "analyzed", or "all"
        """
        print(f"\n{'='*80}")
        print(f"CHUNK STRUCTURE ANALYSIS")
        print(f"{'='*80}\n")

        # Extract and chunk
        text = self.extract_text_from_pdf(pdf_path, use_cache=True)
        raw_chunks = self.chunk_text_by_paragraphs(text)

        if stage in ["raw", "all"]:
            print(f"STAGE 1: Raw chunks (after text splitting)")
            print(f"  Type: List[str]")
            print(f"  Count: {len(raw_chunks)}")
            print(f"  Example structure:")
            if raw_chunks:
                print(f"    chunk[0] = {repr(raw_chunks[0][:100])}...")
                print(f"    Length: {len(raw_chunks[0])} chars\n")

        if stage in ["filtered", "analyzed", "all"]:
            # Filter climate
            if self.run_climate_filter:
                filtered_chunks = self.filter_climate_chunks(raw_chunks)
            else:
                filtered_chunks = [{'text': c} for c in raw_chunks]

            print(f"STAGE 2: Filtered chunks (after climate detection)")
            print(f"  Type: List[Dict]")
            print(f"  Count: {len(filtered_chunks)}")
            print(f"  Example structure:")
            if filtered_chunks:
                print(f"    chunk[0] = {{")
                print(f"      'text': {repr(filtered_chunks[0]['text'][:80])}...,")
                print(f"      'detector_score': {filtered_chunks[0].get('detector_score', 'N/A')}")
                print(f"    }}\n")

        if stage in ["analyzed", "all"]:
            # Analyze
            if self.run_specificity:
                analyzed_chunks = self.analyze_specificity(filtered_chunks[:5])  # Just 5 for demo
            else:
                analyzed_chunks = filtered_chunks[:5]

            print(f"STAGE 3: Analyzed chunks (after specificity)")
            print(f"  Type: List[Dict]")
            print(f"  Example structure:")
            if analyzed_chunks:
                print(f"    chunk[0] = {{")
                print(f"      'text': {repr(analyzed_chunks[0]['text'][:60])}...,")
                print(f"      'detector_score': {analyzed_chunks[0].get('detector_score', 'N/A')},")
                print(f"      'specificity_label': {analyzed_chunks[0].get('specificity_label', 'N/A')},")
                print(f"      'specificity_score': {analyzed_chunks[0].get('specificity_score', 'N/A')}")
                if self.run_sentiment:
                    print(f"      'sentiment_label': {analyzed_chunks[0].get('sentiment_label', 'N/A')},")
                    print(f"      'sentiment_score': {analyzed_chunks[0].get('sentiment_score', 'N/A')}")
                print(f"    }}\n")

In [ ]:

analyzer = ClimateReportAnalyzer(
    cache_dir="cache",
    run_climate_filter=False,
    run_specificity=False,
    run_commitment=False,
    run_sentiment=False
    )

# pdf_path = "data/reports/SSAB/SSAB_Annual_Report_2020_EN.pdf"
# pdf_path = "data/reports/Celsa/007_2020_sustainability_report.pdf"
pdf_path = "data/reports/ArcelorMittal/001_2013_annual_report.pdf"

scores = analyzer.process(pdf_path)

# Inspect specific vs non-specific chunks to validate
# print("\n" + "="*80)
# print("VALIDATION: Inspecting SPECIFIC chunks")
# analyzer.inspect_chunks(pdf_path, n_examples=3, filter_by="specific")

# print("\n" + "="*80)
# print("VALIDATION: Inspecting NON-SPECIFIC chunks")
# analyzer.inspect_chunks(pdf_path, n_examples=3, filter_by="non-specific")

# === EXAMPLE 2: Process all PDFs in directory ===
# analyzer.process(
#     "data/reports_annual",
#     use_cache=True,
#     filter_climate=True,
#     run_sentiment=True,
#     run_commitment=True
# )

# === EXAMPLE 3: Check model labels ===
# from transformers import AutoModelForSequenceClassification
#
# models_to_check = [
#     "climatebert/distilroberta-base-climate-specificity",
#     "climatebert/distilroberta-base-climate-sentiment",
#     "climatebert/distilroberta-base-climate-commitment",
#     "climatebert/distilroberta-base-climate-detector"
# ]
#
# for model_name in models_to_check:
#     model = AutoModelForSequenceClassification.from_pretrained(model_name)
#     print(f"\n{model_name}:")
#     print(f"  Labels: {model.config.id2label}")

Using device: CPU
Config: cache=True, filter=False, spec=False, sent=False, commit=False

PROCESSING SINGLE PDF

Processing: data/reports/ArcelorMittal/001_2013_annual_report.pdf
Loading cached text from cache/001_2013_annual_report_text.txt
Total text length: 1048770 characters
Created 740 chunks

Saved results to cache/001_2013_annual_report_results.json

FINAL RESULTS
File: 001_2013_annual_report.pdf
Company: ArcelorMittal

📊 Component Scores:
  Specificity: 0.250 - Very low specificity (lacks concrete information)

Chunks analyzed: 740
